# 01 — Data Understanding

## Objective

The first stage of the project is to understand the raw dataset before making any cleaning decisions.

We will investigate:

- Dataset dimensions
- Column names
- Data types
- Missing values
- Duplicate records
- Unique values
- Numerical distributions
- Potential data-quality problems

The objective is not to clean the dataset yet, but to understand what we are dealing with.

In [15]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np

from src.utils import get_raw_data_dir

## Load the Dataset

In [16]:
DATA_PATH = (
    get_raw_data_dir()
    / "online_retail_II.xlsx"
)

df = pd.read_excel(DATA_PATH)

print(f"Dataset shape: {df.shape}")

Dataset shape: (525461, 8)


## Inspect the Dataset

In [17]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [18]:
df.tail()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
525456,538171,22271,FELTCRAFT DOLL ROSIE,2,2010-12-09 20:01:00,2.95,17530.0,United Kingdom
525457,538171,22750,FELTCRAFT PRINCESS LOLA DOLL,1,2010-12-09 20:01:00,3.75,17530.0,United Kingdom
525458,538171,22751,FELTCRAFT PRINCESS OLIVIA DOLL,1,2010-12-09 20:01:00,3.75,17530.0,United Kingdom
525459,538171,20970,PINK FLORAL FELTCRAFT SHOULDER BAG,2,2010-12-09 20:01:00,3.75,17530.0,United Kingdom
525460,538171,21931,JUMBO STORAGE BAG SUKI,2,2010-12-09 20:01:00,1.95,17530.0,United Kingdom


In [19]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      525461 non-null  object        
 1   StockCode    525461 non-null  object        
 2   Description  522533 non-null  object        
 3   Quantity     525461 non-null  int64         
 4   InvoiceDate  525461 non-null  datetime64[us]
 5   Price        525461 non-null  float64       
 6   Customer ID  417534 non-null  float64       
 7   Country      525461 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 38.8+ MB


In [20]:
df.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Invoice,525461.0,28816.0,537434.0,675.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
StockCode,525461,4632,85123A,3516,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Description,522533,4681,WHITE HANGING HEART T-LIGHT HOLDER,3549,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,525461.0,NaN,NaN,NaN,10.337667,-9600.0,1.0,3.0,10.0,19152.0,107.42411
InvoiceDate,525461,NaN,NaN,NaN,2010-06-28 11:37:36.845018,2009-12-01 07:45:00,2010-03-21 12:20:00,2010-07-06 09:51:00,2010-10-15 12:45:00,2010-12-09 20:01:00,NaN
Price,525461.0,NaN,NaN,NaN,4.688834,-53594.36,1.25,2.1,4.21,25111.09,146.126914
Customer ID,417534.0,NaN,NaN,NaN,15360.645478,12346.0,13983.0,15311.0,16799.0,18287.0,1680.811316
Country,525461,40,United Kingdom,485852,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Missing Values

In [21]:
missing = (
    df.isna()
    .sum()
    .sort_values(ascending=False)
)

missing

Customer ID    107927
Description      2928
Invoice             0
StockCode           0
Quantity            0
InvoiceDate         0
Price               0
Country             0
dtype: int64

Percentage:

In [22]:
missing_pct = (
    df.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

missing_pct

Customer ID    20.539488
Description     0.557225
Invoice         0.000000
StockCode       0.000000
Quantity        0.000000
InvoiceDate     0.000000
Price           0.000000
Country         0.000000
dtype: float64

## Duplicate Analysis

In [23]:
duplicate_count = df.duplicated().sum()

print(
    f"Exact duplicate rows: {duplicate_count:,}"
)

Exact duplicate rows: 6,865


In [24]:
df.duplicated(
    subset=["Invoice", "StockCode"]
).sum()

np.int64(13335)

## Unique Values

In [25]:
for column in df.columns:
    print(
        f"{column}: "
        f"{df[column].nunique(dropna=True):,} unique values"
    )

Invoice: 28,816 unique values
StockCode: 4,632 unique values
Description: 4,681 unique values
Quantity: 825 unique values
InvoiceDate: 25,296 unique values
Price: 1,606 unique values
Customer ID: 4,383 unique values
Country: 40 unique values


## Initial Data Quality Summary

In [26]:
quality_summary = pd.DataFrame({
    "Metric": [
        "Total Rows",
        "Total Columns",
        "Exact Duplicate Rows",
        "Missing Customer IDs",
        "Negative Quantity Rows",
        "Zero Quantity Rows",
        "Zero Price Rows",
        "Negative Price Rows",
        "Cancelled Invoice Rows",
        "Unique Invoices",
        "Unique Customers",
        "Unique Products",
        "Unique Countries"
    ],

    "Value": [
        len(df),
        df.shape[1],
        df.duplicated().sum(),
        df["Customer ID"].isna().sum(),
        (df["Quantity"] < 0).sum(),
        (df["Quantity"] == 0).sum(),
        (df["Price"] == 0).sum(),
        (df["Price"] < 0).sum(),
        df["Invoice"].astype(str).str.startswith("C").sum(),
        df["Invoice"].nunique(),
        df["Customer ID"].nunique(),
        df["StockCode"].nunique(),
        df["Country"].nunique()
    ]
})

quality_summary

,Metric,Value
0,Total Rows,525461
1,Total Columns,8
2,Exact Duplicate Rows,6865
3,Missing Customer IDs,107927
4,Negative Quantity Rows,12326
5,Zero Quantity Rows,0
6,Zero Price Rows,3687
7,Negative Price Rows,3
8,Cancelled Invoice Rows,10206
9,Unique Invoices,28816


## Initial Findings

The dataset contains approximately 1.07 million transaction records.

Several data-quality issues were identified:

1. A significant proportion of transactions do not contain a Customer ID.
2. Negative quantities appear in the dataset and are interpreted as returns/cancellations.
3. Duplicate records are present.
4. Zero and negative prices occur.
5. Product descriptions contain inconsistencies.
6. Invoice identifiers contain cancellation indicators.

These findings determine the cleaning strategy used in the next stage.